# 🔧 Notebook 03 — Declarative Pipelines Deep Dive

**What you'll learn:**
- Replace 4+ LLM round-trips with a single pipeline call
- MongoDB-style filtering with logical operators (`$and`, `$or`, `$not`)
- Aggregations: group_by, metrics (count, mean, sum, median, stddev)
- `save_as`: store pipeline results as new browsable workspaces
- Nested field access with dot notation

**Why pipelines matter:**
Each tool call = one LLM turn = latency + tokens + cost. A 5-step exploration
(search → filter → sort → limit → select) costs ~$0.10 in API calls and 10+ seconds.
A pipeline does it in ONE call: ~$0.02, 2 seconds.

In [1]:
import json
from ctxtual import Forge, MemoryStore
from ctxtual.utils import paginator, text_search, filter_set, pipeline

forge = Forge(store=MemoryStore())
pager = paginator(forge, "orders")
search = text_search(forge, "orders")
filters = filter_set(forge, "orders")
pipe = pipeline(forge, "orders")

# Simulated e-commerce dataset
import random
random.seed(42)

categories = ["Electronics", "Clothing", "Books", "Home", "Sports", "Food"]
regions = ["US-East", "US-West", "EU-West", "EU-East", "APAC", "LATAM"]
statuses = ["delivered", "shipped", "processing", "cancelled", "returned"]

@forge.producer(workspace_type="orders", toolsets=[pager, search, filters, pipe])
def load_orders() -> list[dict]:
    """Load order data from the warehouse."""
    orders = []
    for i in range(10_000):
        orders.append({
            "order_id": f"ORD-{i:06d}",
            "customer": f"Customer_{i % 500}",
            "category": random.choice(categories),
            "region": random.choice(regions),
            "amount": round(random.uniform(5, 500), 2),
            "quantity": random.randint(1, 20),
            "status": random.choice(statuses),
            "priority": random.choice(["low", "medium", "high"]),
            "year": random.choice([2022, 2023, 2024]),
            "tags": random.sample(["fragile", "express", "gift", "bulk", "subscription"], k=random.randint(0, 3)),
        })
    return orders

ref = load_orders()
wid = ref["workspace_id"]
print(f"Loaded {ref['item_count']:,} orders")
print(f"Schema: {json.dumps(ref.get('item_schema', {}), indent=2)}")

Loaded 10,000 orders
Schema: {
  "type": "object",
  "properties": {
    "order_id": {
      "type": "string"
    },
    "customer": {
      "type": "string"
    },
    "category": {
      "type": "string"
    },
    "region": {
      "type": "string"
    },
    "amount": {
      "type": "number"
    },
    "quantity": {
      "type": "integer"
    },
    "status": {
      "type": "string"
    },
    "priority": {
      "type": "string"
    },
    "year": {
      "type": "integer"
    },
    "tags": {
      "type": "array"
    }
  },
  "required": [
    "order_id",
    "customer",
    "category",
    "region",
    "amount",
    "quantity",
    "status",
    "priority",
    "year",
    "tags"
  ]
}


## 1. Basic Pipeline: Filter → Sort → Limit

In [2]:
# Top 10 highest-value Electronics orders from 2024
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"category": "Electronics", "year": 2024}},
        {"sort": {"field": "amount", "order": "desc"}},
        {"limit": 10},
        {"select": ["order_id", "customer", "amount", "status"]},
    ],
})
print(f"Top 10 Electronics orders in 2024 (from {ref['item_count']:,} total):\n")
for item in result["items"]:
    print(f"  {item['order_id']}  ${item['amount']:>7.2f}  {item['status']:<12}  {item['customer']}")
print(f"\nResult count: {result['count']}")

Top 10 Electronics orders in 2024 (from 10,000 total):

  ORD-008725  $ 499.83  returned      Customer_225
  ORD-000428  $ 499.72  returned      Customer_428
  ORD-009516  $ 499.47  shipped       Customer_16
  ORD-007359  $ 498.70  returned      Customer_359
  ORD-000127  $ 497.98  delivered     Customer_127
  ORD-009882  $ 497.49  delivered     Customer_382
  ORD-009257  $ 493.95  cancelled     Customer_257
  ORD-004477  $ 493.67  returned      Customer_477
  ORD-003272  $ 492.71  returned      Customer_272
  ORD-000611  $ 492.28  processing    Customer_111

Result count: 10


## 2. MongoDB-Style Operators

In [3]:
# Complex filter: high-priority orders over $200 that are NOT delivered
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {
            "priority": "high",
            "amount": {"$gt": 200},
            "status": {"$nin": ["delivered", "cancelled"]},
        }},
        {"sort": {"field": "amount", "order": "desc"}},
        {"limit": 5},
    ],
})
print(f"High-priority, >$200, not delivered/cancelled: {result['count']} found\n")
for item in result["items"]:
    print(f"  {item['order_id']}  ${item['amount']:.2f}  status={item['status']}  region={item['region']}")

High-priority, >$200, not delivered/cancelled: 5 found

  ORD-000428  $499.72  status=returned  region=APAC
  ORD-004239  $499.05  status=shipped  region=EU-East
  ORD-008759  $499.01  status=processing  region=US-East
  ORD-002315  $498.86  status=returned  region=US-East
  ORD-005895  $498.14  status=processing  region=EU-East


In [4]:
# Logical operators: ($or, $and, $not)
# Orders from APAC OR LATAM, in 2024, with "express" in tags
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {
            "$or": [
                {"region": "APAC"},
                {"region": "LATAM"},
            ],
            "year": 2024,
            "tags": {"$contains": "express"},
        }},
        {"count": True},
    ],
})
print(f"APAC/LATAM express orders in 2024: {result['count']}")

APAC/LATAM express orders in 2024: 356


## 3. Aggregations: group_by + Metrics

In [5]:
# Revenue by category
result = forge.dispatch_tool_call("orders_aggregate", {
    "workspace_id": wid,
    "group_by": "category",
    "metrics": {
        "total_revenue": "sum:amount",
        "avg_order": "mean:amount",
        "order_count": "count",
        "max_order": "max:amount",
    },
})
print("=== Revenue by Category ===\n")
print(f"{'Category':<15} {'Orders':>8} {'Total Rev':>12} {'Avg Order':>10} {'Max Order':>10}")
print("-" * 60)
# Sort by total revenue
groups = sorted(result["groups"], key=lambda g: g["total_revenue"], reverse=True)
for g in groups:
    print(f"{g['category']:<15} {g['order_count']:>8} ${g['total_revenue']:>11,.2f} ${g['avg_order']:>9.2f} ${g['max_order']:>9.2f}")
print(f"\nTotal groups: {result['group_count']}")

=== Revenue by Category ===

Category          Orders    Total Rev  Avg Order  Max Order
------------------------------------------------------------
Books               1705 $ 437,718.99 $   256.73 $   499.88
Food                1685 $ 428,594.95 $   254.36 $   499.91
Home                1680 $ 419,923.03 $   249.95 $   499.74
Clothing            1654 $ 418,030.06 $   252.74 $   499.77
Electronics         1679 $ 413,178.80 $   246.09 $   499.91
Sports              1597 $ 395,729.08 $   247.80 $   499.34

Total groups: 6


In [6]:
# Pipeline-based aggregation: group_by inside a pipeline step
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"year": 2024}},
        {"group_by": {
            "field": "region",
            "metrics": {
                "n": "count",
                "revenue": "sum:amount",
                "avg": "mean:amount",
            },
        }},
        {"sort": {"field": "revenue", "order": "desc"}},
    ],
})
print("=== 2024 Revenue by Region ===\n")
for item in result["items"]:
    print(f"  {item['region']:<10} {item['n']:>5} orders  ${item['revenue']:>10,.2f} total  ${item['avg']:>.2f} avg")

=== 2024 Revenue by Region ===

  LATAM        586 orders  $147,639.70 total  $251.94 avg
  US-East      574 orders  $146,719.38 total  $255.61 avg
  EU-East      587 orders  $144,201.12 total  $245.66 avg
  APAC         533 orders  $139,260.28 total  $261.28 avg
  US-West      546 orders  $131,042.79 total  $240.01 avg
  EU-West      524 orders  $127,349.50 total  $243.03 avg


## 4. `flatten` — Explode List Fields

In [7]:
# Tag frequency analysis: flatten tags → group → sort
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"flatten": "tags"},
        {"group_by": {
            "field": "tags",
            "metrics": {"count": "count", "avg_amount": "mean:amount"},
        }},
        {"sort": {"field": "count", "order": "desc"}},
    ],
})
print("=== Tag Frequency Analysis ===\n")
print(f"{'Tag':<15} {'Count':>8} {'Avg Amount':>12}")
print("-" * 40)
for item in result["items"]:
    print(f"{item['tags']:<15} {item['count']:>8} ${item['avg_amount']:>11.2f}")

=== Tag Frequency Analysis ===

Tag                Count   Avg Amount
----------------------------------------
fragile             3083 $     249.87
bulk                3054 $     253.30
subscription        3040 $     251.65
express             2977 $     249.53
gift                2970 $     254.34


## 5. `save_as` — Create Derived Workspaces

In [8]:
# Save high-value orders as a new workspace for further exploration
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"amount": {"$gte": 400}, "status": "delivered"}},
        {"sort": {"field": "amount", "order": "desc"}},
    ],
    "save_as": "premium_orders",
})
print("=== Saved as new workspace ===")
print(f"Status: {result['status']}")
print(f"Items:  {result['item_count']}")
print(f"Schema: {json.dumps(result.get('item_schema', {}), indent=2)}")

=== Saved as new workspace ===
Status: pipeline_result_saved
Items:  395
Schema: {
  "type": "object",
  "properties": {
    "order_id": {
      "type": "string"
    },
    "customer": {
      "type": "string"
    },
    "category": {
      "type": "string"
    },
    "region": {
      "type": "string"
    },
    "amount": {
      "type": "number"
    },
    "quantity": {
      "type": "integer"
    },
    "status": {
      "type": "string"
    },
    "priority": {
      "type": "string"
    },
    "year": {
      "type": "integer"
    },
    "tags": {
      "type": "array"
    }
  },
  "required": [
    "order_id",
    "customer",
    "category",
    "region",
    "amount",
    "quantity",
    "status",
    "priority",
    "year",
    "tags"
  ]
}


In [11]:
# Now paginate the derived workspace!
premium_wid = result["premium_orders"]
page = forge.dispatch_tool_call("orders_paginate", {
    "workspace_id": premium_wid,
    "page": 0,
    "size": 5,
})
print(f"Premium orders workspace: {page['result']['total']} items\n")
for item in page["result"]["items"]:
    print(f"  {item['order_id']}  ${item['amount']:.2f}  {item['region']}")

KeyError: 'premium_orders'

## 6. Advanced: Multi-Sort, Sample, Unique, Text Search

In [12]:
# Deduplicate by customer, keeping highest-value order
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"sort": {"field": "amount", "order": "desc"}},
        {"unique": "customer"},
        {"limit": 10},
        {"select": ["customer", "order_id", "amount"]},
    ],
})
print("=== Top spending customers (1 order each) ===")
for item in result["items"]:
    print(f"  {item['customer']:<15}  {item['order_id']}  ${item['amount']:.2f}")

=== Top spending customers (1 order each) ===
  Customer_260     ORD-006260  $499.91
  Customer_160     ORD-008660  $499.91
  Customer_116     ORD-002116  $499.88
  Customer_225     ORD-008725  $499.83
  Customer_236     ORD-005736  $499.77
  Customer_355     ORD-009355  $499.74
  Customer_428     ORD-000428  $499.72
  Customer_79      ORD-005579  $499.62
  Customer_47      ORD-006047  $499.61
  Customer_289     ORD-006789  $499.61


In [13]:
# Random sample with reproducible seed
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"category": "Books"}},
        {"sample": {"n": 5, "seed": 42}},
        {"select": ["order_id", "customer", "amount"]},
    ],
})
print("=== Random sample of 5 Book orders (seed=42) ===")
for item in result["items"]:
    print(f"  {item['order_id']}  {item['customer']}  ${item['amount']:.2f}")

=== Random sample of 5 Book orders (seed=42) ===
  ORD-007740  Customer_240  $72.35
  ORD-001474  Customer_474  $375.89
  ORD-000293  Customer_293  $57.74
  ORD-009113  Customer_113  $456.31
  ORD-003494  Customer_494  $401.35


In [14]:
# Text search within pipeline
result = forge.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"search": {"query": "Customer_42", "fields": ["customer"]}},
        {"sort": {"field": "amount", "order": "desc"}},
        {"select": ["order_id", "customer", "amount", "category"]},
    ],
})
print(f"=== Orders for Customer_42: {result['count']} found ===")
for item in result["items"][:5]:
    print(f"  {item['order_id']}  ${item['amount']:.2f}  {item['category']}")

=== Orders for Customer_42: 220 found ===
  ORD-000428  $499.72  Electronics
  ORD-006925  $497.90  Sports
  ORD-004929  $497.69  Home
  ORD-002429  $494.47  Home
  ORD-009042  $491.73  Home


## Pipeline Operator Reference

| Operator | Example | Description |
|----------|---------|-------------|
| `filter` | `{"filter": {"year": {"$gte": 2023}}}` | MongoDB-style: `$gt`, `$gte`, `$lt`, `$lte`, `$ne`, `$in`, `$nin`, `$contains`, `$startswith`, `$regex`, `$exists`, `$or`, `$and`, `$not` |
| `search` | `{"search": {"query": "ML", "fields": ["title"]}}` | Case-insensitive text search |
| `sort` | `{"sort": {"field": "score", "order": "desc"}}` | Single or multi-field sort |
| `select` | `{"select": ["title", "author"]}` | Keep only listed fields |
| `exclude` | `{"exclude": ["raw_blob"]}` | Remove listed fields |
| `limit` | `{"limit": 10}` | First N items |
| `skip` | `{"skip": 5}` | Skip first N |
| `slice` | `{"slice": [5, 15]}` | Items `[5:15]` |
| `sample` | `{"sample": {"n": 10, "seed": 42}}` | Random sample |
| `unique` | `{"unique": "author"}` | Deduplicate by field |
| `flatten` | `{"flatten": "tags"}` | Expand list fields |
| `group_by` | `{"group_by": {"field": "cat", "metrics": {"n": "count"}}}` | Aggregation |
| `count` | `{"count": true}` | Terminal: return `{"count": N}` |

**Next:** [04_openai_agent_loop.ipynb](04_openai_agent_loop.ipynb) — full agent loop with OpenAI SDK